# Small Language Model Project

In this notebook I am building a small language model from scratch. The goal is to keep the number of parameters around 50 to 60 million. The model will be trained on short stories and it should be able to generate new short stories on its own.

## Task 1: Loading the Dataset

A language model learns by predicting the next word given all the words before it. It does this one token at a time, which is called next token prediction.

I used the TinyStories dataset from Hugging Face because it contains short and simple stories. This makes it a good fit for training a small model since the vocabulary and sentence structures are not too complex.

In [ ]:
!pip install datasets

from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")
print(ds)
print(ds["train"]["text"][0])

## Task 2: Tokenizing the Data

Computers cannot work with raw text so I need to convert each story into a list of numbers called tokens. I am using tiktoken with the GPT-2 encoding for this.

First I tested it on a short sentence to check that encoding and decoding round-trips correctly.

In [ ]:
!pip install tiktoken

import tiktoken

enc = tiktoken.get_encoding("gpt2")
sample_text = "I love building AI models"
ids = enc.encode_ordinary(sample_text)
print(ids)
print(enc.decode(ids))

In [ ]:
import os
import numpy as np
from tqdm.auto import tqdm


def process(example):
    ids = enc.encode_ordinary(example["text"])
    ids.append(enc.eot_token)
    return {"ids": ids, "len": len(ids)}


if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=["text"],
        num_proc=8,
    )
    for split, dset in tokenized.items():
        arr_len = np.sum(dset["len"], dtype=np.uint64)
        filename = "train.bin" if split == "train" else "val.bin"
        dtype = np.uint16
        arr = np.memmap(filename, dtype=dtype, mode="w+", shape=(arr_len,))
        total_batches = 1024
        idx = 0
        batch_size = 5000
        for i in tqdm(range(0, len(dset), batch_size)):
            batch = dset[i : i + batch_size]
            arr_batch = np.array([tok for story in batch["ids"] for tok in story], dtype=np.uint16)
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()

In [ ]:
train_data = np.memmap("train.bin", dtype=np.uint16, mode="r")
print(len(train_data))
print(enc.decode(train_data[:50].tolist()))

## Task 3: Building the Data Loader

The model needs input and target pairs to learn from. For each sequence of tokens the target is just the same sequence shifted one position to the right. That way the model learns to predict what comes next at every position.

In [ ]:
import torch

batch_size = 32
block_size = 128
device = "cuda" if torch.cuda.is_available() else "cpu"


def get_batch(split):
    if split == "train":
        data = np.memmap("train.bin", dtype=np.uint16, mode="r")
    else:
        data = np.memmap("val.bin", dtype=np.uint16, mode="r")
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


x, y = get_batch("train")
print(x.shape, y.shape)
print(x[0][:10])
print(y[0][:10])

## Task 4: Building the Transformer Model

This is the main part of the assignment. I built a small GPT-style transformer in PyTorch. The model has token embeddings, position embeddings, a stack of transformer blocks, and a linear head that outputs a score for each token in the vocabulary.

In [ ]:
from dataclasses import dataclass


@dataclass
class GPTConfig:
    vocab_size: int = 50257
    block_size: int = 128
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.1
    bias: bool = True

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math


class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = LayerNorm(config.n_embd, config.bias)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        tok_embeddings = self.tok_emb(idx)
        pos_embeddings = self.pos_emb(pos)
        x = self.drop(tok_embeddings + pos_embeddings)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
config = GPTConfig()
model = GPT(config).to(device)
dummy_input = torch.randint(0, config.vocab_size, (2, config.block_size)).to(device)
logits, loss = model(dummy_input)
print(logits.shape)
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

## Task 5: Setting Up the Optimizer and Scheduler

I used AdamW as the optimizer because it handles weight decay separately from the gradient update, which tends to give better results than standard Adam. I also added a cosine annealing scheduler so the learning rate decreases smoothly over training.

In [ ]:
learning_rate = 3e-4
max_iters = 20000
eval_iters = 500

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iters)

print(optimizer.param_groups[0]["lr"])
optimizer.step()
scheduler.step()
print(optimizer.param_groups[0]["lr"])

## Task 6: Training the Model

Each training step picks a random batch, runs it through the model, computes cross-entropy loss against the targets, and updates the weights with backpropagation. Every 500 steps I evaluate on the validation set and save the model if the validation loss improves.

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
best_val_loss = float("inf")
best_model_path = "best_model_params.pt"
train_loss_list = []
val_loss_list = []

for step in tqdm(range(max_iters)):
    if step % eval_iters == 0:
        losses = estimate_loss()
        print(f"step {step} train loss {losses['train']:.4f} val loss {losses['val']:.4f}")
        train_loss_list.append(losses["train"])
        val_loss_list.append(losses["val"])
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            torch.save(model.state_dict(), best_model_path)
    X, Y = get_batch("train")
    logits, loss = model(X, Y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
    optimizer.step()
    scheduler.step()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_loss_list, label="train loss")
plt.plot(val_loss_list, label="val loss")
plt.xlabel(f"every {eval_iters} steps")
plt.ylabel("loss")
plt.legend()
plt.show()

## Task 7: Evaluating the Model

Loss alone is hard to interpret, so I also computed perplexity which is just exp(validation loss). A lower perplexity means the model is more confident about its predictions. I also plotted the train and validation loss curves to check for overfitting.

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
val_losses = estimate_loss()
perplexity = math.exp(val_losses["val"])
print(f"validation loss is {val_losses['val']:.4f}")
print(f"perplexity is {perplexity:.4f}")

Looking at the loss curves, the train loss and validation loss both go down together and the validation loss does not start going back up. So the model does not seem to be overfitting. The generated samples below also look mostly readable which is a good sign.

## Task 8: Generating Text

I used the trained model to generate new short stories starting from the prompt "Once upon a time". I tried two temperature values to see how they affect the output. A lower temperature makes the model pick safer, more common words, while a higher temperature introduces more randomness and variety.

In [ ]:
model = GPT(config).to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
prompt = "Once upon a time"
context = torch.tensor(enc.encode_ordinary(prompt)).unsqueeze(0).to(device)

In [ ]:
output = model.generate(context, max_new_tokens=150, temperature=0.7)
print(enc.decode(output[0].tolist()))

In [ ]:
output = model.generate(context, max_new_tokens=150, temperature=1.2)
print(enc.decode(output[0].tolist()))

With a lower temperature of 0.7 the story stays more on topic and the sentences are more sensible. With a higher temperature of 1.2 the output becomes more random and sometimes the sentences do not fully make sense, but the words used are more varied.